# RECOLECCIÓN Y LIMPIEZA DE DATA

En este notebook realizaremos la recolección y limpieza de los datos que se están guardando diariamente en las carpetas buses_outputs y metro_outputs. Con ello, uniremos los datos correspondientes a las micros con los del archivo JSON guardado en la carpeta data.

In [6]:
# librerías a utilizar
import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timedelta, date

In [7]:
# Creamos funcion que añada nuevos datos al dataframe unificado 
def anadir_datos(nuevos_datos, ruta):
    # Dataframe de los datos nuevos
    df_nuevos_datos = pd.concat(nuevos_datos, ignore_index=True)
    # Si ya tenemos datos en dicha fecha:
    if ruta.exists():
        # Abrimos los datos ya existentes
        datos_guardados = pd.read_csv(ruta)
        # Les añadimos los nuevos datos recolectados
        nuevo_csv = pd.concat([datos_guardados, df_nuevos_datos], ignore_index=True)
        # volvemos a guardar en la ruta que estaban los datos ya existentes
        nuevo_csv.to_csv(ruta, index=False, encoding='utf-8')
    else:
        # si no teníamos datos en dicha fecha
        nuevo_csv = df_nuevos_datos
        # agregamos los datos a la ruta 
        nuevo_csv.to_csv(ruta, index=False, encoding='utf-8')
    return nuevo_csv

In [8]:
# Creamos las rutas a la carpeta de la micro y metro:
carpeta_bus = Path('buses_outputs')
carpeta_metro = Path('metro_outputs')
# Creamos las carpetas para guardar los datos:
csv_unificado_micro = Path('csv_unificado_micro')
csv_unificado_micro.mkdir(exist_ok=True)
csv_unificado_metro = Path('csv_unificado_metro')
csv_unificado_metro.mkdir(exist_ok=True)
# Determinamos ruta para CSV's unificados
ruta_unificada_micro = csv_unificado_micro / 'datos_unificados_micro.csv'
ruta_unificada_metro = csv_unificado_metro / 'datos_unificados_metro.csv'
# Determinamos formato de tiempo que tienen los distintos csv
tiempo_hora_minuto = '%d-%m-%Y-%H-%M'
# Determinamos fecha inicial de extracción
fecha_inicio = date(2025, 10, 12)
print(f'La fecha de inicio para la extracción de datos es: {fecha_inicio}')


La fecha de inicio para la extracción de datos es: 2025-10-12


In [9]:
# Fecha en la que empezamos la extracción
fecha_actual = fecha_inicio
# Para los datos de la micro
info_df_diario = []
# Enlistamos los datos de 'buses_outputs'
recorrer = list(carpeta_bus.glob('*csv'))
# diccionario micros para ver si faltan agregar datos de un día
micro_dict = dict()
# Los recorremos para empezar a crear los DataFrame
for i in range(len(recorrer)): 
    # Ponemos en el formato de fecha de los datos extraídos: día, mes y año
    fecha_actual_formato = fecha_actual.strftime('%d-%m-%Y')
    print(f'Recolectando datos de las micros del día {fecha_actual_formato}')
    # Nos quedamos con la fecha del archivo que estamos leyendo
    archivo_fecha = recorrer[i].stem[6:]
    try: 
        # Si el archivo de esta iteración es valor de la llave de la fecha actual
        if archivo_fecha in micro_dict[fecha_actual_formato]:
            continue
        else:
            # si no lo es, lo añadimos
            micro_dict[fecha_actual_formato].add(archivo_fecha)
    # Si esa llave no existía (no habíamos revisado nada de ese día), la creamos
    except KeyError:
        # La creamos en un set para hacer más fácil su búsqueda después
        micro_dict[fecha_actual_formato] = {f'{archivo_fecha}'}
    # Convertimos la fecha del archivo al formato tiempo_hora_minuto
    hora_minuto = datetime.strptime(archivo_fecha, tiempo_hora_minuto)
    # Leemos el csv actual
    print(f'Recolectando datos de las: {hora_minuto.hour:02d}:{hora_minuto.minute:02d}')
    dataframe_archivo = pd.read_csv(recorrer[i], encoding='utf-8')
    # Creamos una columna que incluya la hora y el minuto
    dataframe_archivo['Hour_minute'] = f'{hora_minuto.hour:02d}:{hora_minuto.minute:02d}'
    # añadimos el df creado a nuestra lista de información por día
    info_df_diario.append(dataframe_archivo)
    try:
        # Si la fecha actual no está en el siguiente elemento, se acabaron los datos del
        # día y debemos guardar todo lo recolectado
        if fecha_actual_formato not in recorrer[i+1].stem:
            # creamos la ruta donde guardaremos los datos
            anadir_datos(info_df_diario, ruta_unificada_micro)
            # Vaciamos la info del día pues vamos a cambiarlo
            info_df_diario = []
            # Pasamos al día siguiente
            fecha_actual += timedelta(days=1)

    # Si llegamos al final de la lista no se podrá hacer i+1
    except IndexError as error:
        # creamos ruta para el último día de datos
        anadir_datos(info_df_diario, ruta_unificada_micro)
        # Vaciamos info del día
        info_df_diario = []

Recolectando datos de las micros del día 12-10-2025
Recolectando datos de las: 19:01
Recolectando datos de las micros del día 12-10-2025
Recolectando datos de las: 20:00
Recolectando datos de las micros del día 12-10-2025
Recolectando datos de las: 22:00
Recolectando datos de las micros del día 13-10-2025
Recolectando datos de las: 06:00
Recolectando datos de las micros del día 13-10-2025
Recolectando datos de las: 08:00
Recolectando datos de las micros del día 13-10-2025
Recolectando datos de las: 10:00
Recolectando datos de las micros del día 13-10-2025
Recolectando datos de las: 12:00
Recolectando datos de las micros del día 13-10-2025
Recolectando datos de las: 14:00
Recolectando datos de las micros del día 13-10-2025
Recolectando datos de las: 16:00
Recolectando datos de las micros del día 13-10-2025
Recolectando datos de las: 18:00
Recolectando datos de las micros del día 13-10-2025
Recolectando datos de las: 20:00
Recolectando datos de las micros del día 13-10-2025
Recolectando 

In [ ]:
# Añadimos los datos diarios del metro
df_metro_diario = []
# Partimos con la fecha de inicio
fecha_actual_metro = fecha_inicio
datos = list(carpeta_metro.glob('*csv'))
# creamos diccionario para ver si faltan agregar los datos de un día
metro_dict = dict()
for i in range(len(datos)):
    # Trabajamos la fecha según como salen los datos de carpeta_metro
    formato_metro = fecha_actual_metro.strftime('%d-%m-%Y')
    print(f'Recolectando datos del metro del día: {formato_metro}')
    fecha_a_revisar = datos[i].stem[6:]
    try: 
        if fecha_a_revisar in metro_dict[formato_metro]:
            continue
        else:
            metro_dict[formato_metro].add(fecha_a_revisar)
    except KeyError:
        metro_dict[formato_metro] = {f'{fecha_a_revisar}'}
    hora_minuto = datetime.strptime(fecha_a_revisar, tiempo_hora_minuto)
    print(f'Recolectando datos de las: {hora_minuto.hour:02d}:{hora_minuto.minute:02d}')
    dataframe_actual = pd.read_csv(datos[i], encoding='utf-8')
    dataframe_actual['Hora_minuto'] = f'{hora_minuto.hour:02d}:{hora_minuto.minute:02d}'
    df_metro_diario.append(dataframe_actual)
    try:
        # Si la fecha actual no está en el siguiente elemento, se acabaron los datos del
        # día y debemos guardar todo lo recolectado
        if formato_metro not in datos[i+1].stem:
            # creamos la ruta donde guardaremos los datos
            anadir_datos(df_metro_diario, ruta_unificada_metro)
            df_metro_diario = []
            # Sumamos un día a la fecha actual
            fecha_actual_metro += timedelta(days=1)
    # Si llegamos al final de la lista no se podrá hacer i+1
    except IndexError as error:
        # creamos ruta para el último día de datos
        anadir_datos(df_metro_diario, ruta_unificada_metro)
        df_metro_diario = []   

In [ ]:
# Leemos los csv generados
micro_data = pd.read_csv(ruta_unificada_micro)
metro_data = pd.read_csv(ruta_unificada_metro)
# Eliminamos columnas con coordenadas incorrectas
micro_data.drop(columns=['X', 'Y'], inplace=True)
micro_data.head(3)

In [ ]:
# Abrimos el archivo donde tenemos información importantes de los pasajeros
ruta_json = Path('data', 'Paraderos-Santiago-Chile.geojsonl.json')
# Guardamos su informacion en una lista
datos = []
with open(ruta_json, 'r', encoding='utf-8-sig')as f:
    for archivo in f:
        datos.append(json.loads(archivo.strip()))
# creamos un datafraem con los datos de la lista
df_paradero = pd.json_normalize(datos)
# Renombramos columnas que estarán en nuestro DataFrame
df_paradero = df_paradero.rename(columns={'properties.ID': 'id', 
                            'properties.CODINFRA': 'codinfra',
                            'properties.SIMT': 'bus_stop_code',
                            'properties.COMUNA': 'comuna',
                            'properties.NOMBRE_PAR': 'nombre_par',
                            'properties.NSERVICIOS': 'n_servicios',
                            'properties.SERVICIOS': 'servicios',
                            'geometry.coordinates': 'coordinates'})
# Seleccionamos dichas columnas
df_paradero = df_paradero[['id', 'codinfra', 'comuna', 'bus_stop_code', 'nombre_par', 'n_servicios', 'servicios', 'coordinates']]
# Reemplazamos nombres de comunas pues el archivo venía con un encoding incorrecto
df_paradero['comuna'] = df_paradero['comuna'].replace(
    {'CONCHAL�': 'CONCHALÍ',
    'ESTACI�N CENTRAL': 'ESTACIÓN CENTRAL',
    'MAIP�': 'MAIPÚ',
    'PE�ALOL�N': 'PEÑALOLÉN',
    'SAN JOAQU�N': 'SAN JOAQUÍN',
    'SAN RAM�N': 'SAN RAMÓN',
    '�U�OA': 'ÑUÑOA'
    })
df_paradero.head()

In [ ]:
# Unimos por paradero el JSON y nuestro archivo de micros.
micro_paradero = pd.merge(micro_data, df_paradero, on= 'bus_stop_code', how='inner')
micro_paradero.head()

In [ ]:
# Dejamos las coordenadas en columnas Lan y Lon. Eliminamos columna coordinates
micro_paradero[['lan', 'lon']] = pd.DataFrame(micro_paradero['coordinates'].tolist(), index=micro_paradero.index)
micro_paradero.drop(columns=['coordinates'], inplace= True)
micro_paradero.head()

Podemos notar al revisar este DataFrame, en la columna 'servicios', que todos los códigos de micro tienen una letra al final (Sabemos que puede ser N, C, E, V) pero también las hay en casos que no corresponden. Tomemos un paradero como ejemplo:

In [ ]:
# PARADERO DE EJEMPLO
paradero_ejemplo = micro_paradero[micro_paradero['bus_stop_code'] == 'PJ607'].head(1)
print(f'Los recorridos que pasan por este paradero en la fila: {paradero_ejemplo['servicios']}')
print('Esto no tiene sentido, no existe la J01R y la ruta corta corresponde a la J01c')

En consecuencia, es necesario que los datos correspondientes a los recorridos sean los correctos:

In [ ]:
micro_limpio = micro_paradero.copy()
micro_limpio['servicios'] = micro_limpio['servicios'].str.replace(r'([IR])(?=;|$)', '', regex=True)
micro_exp = micro_limpio['servicios'].str.split(';', expand=True)

micro_exp = micro_exp.apply(lambda x: pd.Series(x.unique()), axis=1)
micro_exp = micro_exp.drop([21], axis=1)
micro_exp

In [ ]:
recorridos_limpios = pd.concat([micro_limpio, micro_exp], axis=1)
# Borramos columna con los datos incorrectos. Ahora cada dato tendrá una columna propia, enumerada del 1 al 20 (algunas son NaN)
recorridos_limpios.drop(columns=['servicios'], inplace=True)
# Guardamos la información en un dataframe
carpeta_data = Path('data')
ruta_paradero_limpio = carpeta_data / 'paraderos_limpios.csv'
recorridos_limpios.to_csv(ruta_paradero_limpio, index=False, encoding='utf-8')


In [ ]:
micro_limpio.head(3)

In [ ]:
recorridos_limpios = pd.concat([micro_limpio, micro_exp], axis=1)
recorridos_limpios.columns
recorridos_limpios